# Zero Gaze — Free Kaggle Runtime (T4 GPU / CPU)

> **Zero Gaze** is an autonomous agent that reads an arXiv paper, extracts empirical benchmark claims, discovers existing implementations, synthesizes a baseline experiment plan, and executes it with interactive human confirmation.

### Instructions
1. **Internet Access**: Ensure **Internet: On** is selected in the notebook sidebar settings.
2. **OpenRouter Key**: Set your `OPENROUTER_API_KEY` in the environment or input prompt below.

In [ ]:
# Cell 1: Environment Setup & Dependency Installation
!pip install -q pydantic langgraph langchain-openai pymupdf4llm httpx ipywidgets
import os, sys
print('Environment initialized successfully.')

In [ ]:
# Cell 2: Configure API Keys & Target Paper
import getpass
if not os.environ.get('OPENROUTER_API_KEY'):
    api_key = getpass.getpass('Enter your OpenRouter API Key: ')
    os.environ['OPENROUTER_API_KEY'] = api_key

PAPER_TARGET = '2106.09685'  # LoRA: Low-Rank Adaptation of Large Language Models
print(f'Configured paper target: {PAPER_TARGET}')

In [ ]:
# Cell 3: Ingest Paper & Discover Artifacts
from zero_gaze.ingestion import PaperIngestionEngine
from zero_gaze.discovery import ArtifactDiscoveryEngine

print('1. Ingesting paper...')
ingestion = PaperIngestionEngine()
paper = ingestion.ingest(PAPER_TARGET)
print(f'   Ingested: {paper.title} ({len(paper.full_text_markdown)} chars, {len(paper.section_headers)} sections)')

print('2. Discovering code and datasets...')
discovery = ArtifactDiscoveryEngine()
code = discovery.discover(paper.paper_id, paper.title)
print(f'   Discovered Code: {code.repo_url} (Status: {code.status.value}, Stars: {code.stars})')

In [ ]:
# Cell 4: Extract Claims & Synthesize Replication Plan
from zero_gaze.llm import ClaimExtractor, ModelGateway, ReplicationPlanner

gateway = ModelGateway()
extractor = ClaimExtractor(gateway=gateway)
planner = ReplicationPlanner(gateway=gateway)

print('Extracting benchmark claims...')
claims = extractor.extract_claims(paper.full_text_markdown, paper.title)
for c in claims.claims:
    print(f' - {c.benchmark_name}: {c.target_metric} = {c.paper_value} ({c.baseline_algorithm})')

print('Synthesizing replication plan...')
plan = planner.plan_replication(paper=paper, claims=claims.claims, code_resource=code, target_hardware='gpu_t4')
print(f'Execution Command: {plan.execution_command}')
print(f'Estimated Runtime: {plan.estimated_runtime_minutes} min')

In [ ]:
# Cell 5: Human-in-the-Loop In-Notebook Approval Gate
import ipywidgets as widgets
from IPython.display import display

print('=' * 60)
print('HUMAN-IN-THE-LOOP APPROVAL GATE')
print('=' * 60)
print(f'Paper: {paper.title}')
print(f'Command: {plan.execution_command}')

btn_approve = widgets.Button(description='Approve & Run', button_style='success', icon='check')
btn_abort = widgets.Button(description='Abort', button_style='danger', icon='times')
output = widgets.Output()

approval_state = {'approved': False, 'decided': False}

def on_approve(b):
    approval_state['approved'] = True
    approval_state['decided'] = True
    with output:
        print('Plan Approved. Proceeding to execution...')

def on_abort(b):
    approval_state['approved'] = False
    approval_state['decided'] = True
    with output:
        print('Execution aborted by operator.')

btn_approve.on_click(on_approve)
btn_abort.on_click(on_abort)
display(widgets.HBox([btn_approve, btn_abort]), output)

In [ ]:
# Cell 6: Execute Baseline in Ephemeral Sandbox
from zero_gaze.execution import SandboxRunner

if approval_state.get('approved', True):
    print('Executing replication script in sandbox...')
    sandbox = SandboxRunner(timeout_seconds=120.0)
    res = sandbox.execute_script(plan.baseline_script)
    print(f'Exit Code: {res.exit_code} (Success: {res.success}, Time: {res.runtime_seconds}s)')
    print('Stdout:\n', res.stdout[:500] if res.stdout else 'None')
    print('Output Metrics:', res.output_metrics)
else:
    print('Execution was aborted by operator.')